# 📐 傅里叶变换、Plancherel 定理与二维 FFT 图像处理

> 本 Notebook 面向**信号与图像处理方向**，通过一步步可视化实验，加强对于离散和连续傅里叶变换的理解：
>
> * 傅里叶变换 / 逆变换
> * Plancherel 定理（能量守恒）
> * 卷积 ↔ 频域乘法
> * 二维 FFT 在图像处理中的核心作用

---

## 0. 学习目标（Learning objectives）

完成本 Notebook 后，你将能够：

* 直观理解 **时域 / 空间域** 与 **频域** 的关系
* 用数值实验验证 **Plancherel 定理**
* 理解为什么“频域滤波 = 空间域卷积”
* 在二维图像上使用 FFT 进行低通 / 高通处理

---

## 1. 一维信号：从时域到频域

### 1.1 构造测试信号

我们先构造一个由**低频 + 高频**组成的一维信号。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 时间轴
N = 1024
x = np.linspace(0, 1, N, endpoint=False)

# 信号：低频 + 高频
f = np.sin(2*np.pi*5*x) + 0.5*np.sin(2*np.pi*40*x)

plt.figure()
plt.plot(x, f)
plt.title("Signal in time domain")
plt.xlabel("x")
plt.ylabel("f(x)")
plt.show()

📌 **思考**：

* 你能从图中分辨出两个频率成分吗？

---

### 1.2 傅里叶变换（FFT）

In [ ]:
F = np.fft.fft(f)
frequencies = np.fft.fftfreq(N, d=1/N)

plt.figure()
plt.stem(frequencies[:N//2], np.abs(F)[:N//2], use_line_collection=True)
plt.title("Magnitude of Fourier transform")
plt.xlabel("frequency")
plt.ylabel("|F(ξ)|")
plt.show()

📌 **观察**：

* 峰值出现在哪些频率？
* 这与你构造信号时的频率是否一致？

---

## 2. Plancherel 定理（能量守恒）

### 2.1 理论回顾（连续情形）

$$\int_{\mathbb R} |f(x)|^2\,dx
= \frac{1}{2\pi}\int_{\mathbb R} |\widehat f(\xi)|^2\,d\xi$$

➡️ **能量在时域和频域中是一样的**。

---

### 2.2 数值验证（离散版本）

1. NumPy的fft使用非归一化定义：
- 正变换：$F[k] = \sum_{n=0}^{N-1} f[n] e^{-i2\pi kn/N}$
- 逆变换：$f[n] = \frac{1}{N} \sum_{k=0}^{N-1} F[k] e^{i2\pi kn/N}$
2. Plancherel定理的离散形式：
$$\sum_{n=0}^{N-1} |f[n]|^2 = \frac{1}{N} \sum_{k=0}^{N-1} |F[k]|^2$$

In [ ]:
energy_time = np.sum(np.abs(f)**2)
energy_freq = np.sum(np.abs(F)**2) / N

energy_time, energy_freq

📌 **结论**：

* 两个数值是否接近？
* 为什么频域需要除以 `N`？（提示：FFT 的归一化）

✏️ **练习 1（填空）**：

> Plancherel 定理说明：信号的 ______ 在时域和 ______ 中保持不变。

---

## 3. 卷积 vs 频域乘法

### 3.1 构造平滑核（Gaussian）

In [ ]:
sigma = 0.02
gaussian = np.exp(-((x-0.5)**2)/(2*sigma**2))
gaussian /= gaussian.sum()

plt.figure()
plt.plot(x, gaussian)
plt.title("Gaussian kernel")
plt.show()

---

### 3.2 空间域/时间域卷积

convolve(f, gaussian)

In [ ]:
from numpy import convolve

f_smooth = convolve(f, gaussian, mode='same')

plt.figure()
plt.plot(x, f, label="original")
plt.plot(x, f_smooth, label="smoothed")
plt.legend()
plt.show()

---

### 3.3 频域乘法

F * G，然后逆变换

In [ ]:
G = np.fft.fft(gaussian)
F_smooth_freq = F * G
f_smooth2 = np.real(np.fft.ifft(F_smooth_freq))

np.max(np.abs(f_smooth - f_smooth2))

📌 **结论**：

$$f * g \quad \longleftrightarrow \quad \widehat f \cdot \widehat g$$

✏️ **练习 2**：

> 为什么在频域中做乘法，比在空间域中做卷积更高效？

---

## 4. 二维 FFT：进入图像处理

### 4.1 读取并显示图像

In [ ]:
from skimage import data
from skimage.color import rgb2gray

image = data.camera()

plt.figure(figsize=(4,4))
plt.imshow(image, cmap='gray')
plt.title("Original image")
plt.axis('off')
plt.show()

---

### 4.2 二维傅里叶变换

In [ ]:
F2 = np.fft.fft2(image)
F2_shifted = np.fft.fftshift(F2)

plt.figure(figsize=(4,4))
plt.imshow(np.log(1 + np.abs(F2_shifted)), cmap='gray')
plt.title("Log magnitude of FFT")
plt.axis('off')
plt.show()

📌 **解释**：

* 中心：低频（轮廓、光照）
* 边缘：高频（纹理、边缘）

---

## 5. 频域滤波（低通 / 高通）

### 5.1 构造低通掩膜

1. 创建圆形掩膜（中心为1，边缘为0）
2. 频域相乘：F2_shifted * mask
3. 逆变换回空间域

In [ ]:
rows, cols = image.shape
crow, ccol = rows//2, cols//2

mask = np.zeros_like(image)
radius = 50
for i in range(rows):
    for j in range(cols):
        if (i-crow)**2 + (j-ccol)**2 < radius**2:
            mask[i,j] = 1

plt.figure(figsize=(4,4))
plt.imshow(mask, cmap='gray')
plt.title("Low-pass mask")
plt.axis('off')
plt.show()

---

### 5.2 应用滤波并逆变换

In [ ]:
F2_filtered = F2_shifted * mask
image_filtered = np.real(np.fft.ifft2(np.fft.ifftshift(F2_filtered)))

plt.figure(figsize=(4,4))
plt.imshow(image_filtered, cmap='gray')
plt.title("Low-pass filtered image")
plt.axis('off')
plt.show()

✏️ **练习 3**：

1. 把低通改成高通（`1-mask`），观察图像变化
2. 改变 `radius`，分析模糊程度

---

## 6. Plancherel 在工程中的意义（总结）

* 📡 **信号能量守恒**：滤波不会“凭空创造能量”
* 🖼 **图像处理**：频域操作 = 稳定、可控
* 🎧 **通信 / 音频**：噪声分析、频带分离
* 🤖 **机器学习**：卷积神经网络的数学基础

---

##  总结

> **Plancherel 定理告诉我们：频域不是“另一种世界”，而是同一信号的另一种坐标表达。**

---

📘 *下一步建议*：

* 非理想滤波 vs Gaussian 滤波
* Gibbs 现象（频域截断）
* FFT 在 CNN 中的加速实现